# 🏆 Amazon ML Challenge 2026 — Business Entity Resolution
## Full Pipeline: EDA → Blocking → Feature Engineering → Train → Validate → Test Inference

**Metric:** Macro F0.5 (precision weighted 2× recall)

**Pipeline:**
1. Data loading & EDA
2. Text normalization
3. Multi-key blocking (SQLite-backed index)
4. Pairwise feature engineering
5. XGBoost training + F0.5 threshold tuning
6. Validation on held-out S1 slice
7. Full test inference → `output/matching_results.tsv` + `output/candidate_pairs.tsv`
8. Format validation

---
## 0. Setup & Path Discovery

In [ ]:
from pathlib import Path

TRAIN_DB = Path(r'd:\ML\amazon-ml-challenge-2026\Dataset\artifacts\train_index.db')

try:
    if TRAIN_DB.exists():
        TRAIN_DB.unlink()
        print(f'🗑  Deleted: {TRAIN_DB}')
    else:
        print('File does not exist — nothing to delete.')
except PermissionError:
    print('Still locked! Open Task Manager → Details → kill all python.exe → try again.')


Still locked! Open Task Manager → Details → kill all python.exe → try again.


: 

In [1]:
import os, re, sys, csv, json, time, math, random, pickle, sqlite3, hashlib, unicodedata, warnings
from pathlib import Path
from difflib import SequenceMatcher
from collections import defaultdict
from typing import Dict, Set, List, Tuple

import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

print(f'Python {sys.version}')
print(f'pandas {pd.__version__}  numpy {np.__version__}')

# ── Resolve paths automatically ───────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
SEARCH_ROOTS = [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent / 'Dataset']
DATA_CANDIDATES = (
    [r / 'student_resource' / 'dataset' for r in SEARCH_ROOTS]
    + [r / 'Dataset' / 'student_resource' / 'dataset' for r in SEARCH_ROOTS]
    + [r / 'dataset' for r in SEARCH_ROOTS]
)
DATA = next(
    (p for p in DATA_CANDIDATES if (p / 'train' / 'train_source1.tsv').is_file()),
    None
)
if DATA is None:
    raise FileNotFoundError('Dataset not found. Checked: ' + ', '.join(str(p) for p in DATA_CANDIDATES))

TRAIN = DATA / 'train'
TEST  = DATA / 'test'
REPO  = DATA.parent.parent
OUT   = REPO / 'output'
ARTIFACTS = REPO / 'artifacts'
OUT.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

print(f'\n📂 Dataset root : {DATA}')
print(f'📂 Output dir   : {OUT}')
print(f'📂 Artifacts dir: {ARTIFACTS}')

Python 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
pandas 2.3.3  numpy 1.26.4

📂 Dataset root : d:\ML\amazon-ml-challenge-2026\Dataset\student_resource\dataset
📂 Output dir   : d:\ML\amazon-ml-challenge-2026\Dataset\output
📂 Artifacts dir: d:\ML\amazon-ml-challenge-2026\Dataset\artifacts


---
## 1. Exploratory Data Analysis

In [2]:
def load_tsv(path, usecols=None, nrows=None):
    kwargs = dict(sep='\t', dtype=str, encoding='utf-8', na_filter=False)
    if usecols: kwargs['usecols'] = usecols
    if nrows:   kwargs['nrows']   = nrows
    return pd.read_csv(path, **kwargs)

print('📊 File sizes:')
files_info = {
    'train_s1':  TRAIN / 'train_source1.tsv',
    'train_s2':  TRAIN / 'train_source2.tsv',
    'train_s3':  TRAIN / 'train_source3.tsv',
    'train_gt':  TRAIN / 'train_ground_truth.tsv',
    'test_s1':   TEST  / 'test_source1.tsv',
    'test_s2':   TEST  / 'test_source2.tsv',
    'test_s3':   TEST  / 'test_source3.tsv',
}
for name, path in files_info.items():
    mb = path.stat().st_size / 1e6
    with path.open(encoding='utf-8') as f:
        rows = sum(1 for _ in f) - 1
    print(f'  ✅ {name:12s}  {rows:>10,} rows   {mb:>7.1f} MB')

📊 File sizes:
  ✅ train_s1       2,206,821 rows     210.1 MB
  ✅ train_s2       5,034,616 rows     489.3 MB
  ✅ train_s3       5,285,603 rows     503.7 MB
  ✅ train_gt       2,206,821 rows     127.0 MB
  ✅ test_s1        1,732,544 rows     175.0 MB
  ✅ test_s2        4,887,273 rows     509.5 MB
  ✅ test_s3        5,082,316 rows     506.0 MB


In [3]:
EDA_N = 50_000
print(f'Loading {EDA_N:,} rows from each source for EDA...')
s1_sample = load_tsv(TRAIN / 'train_source1.tsv', nrows=EDA_N)
s2_sample = load_tsv(TRAIN / 'train_source2.tsv', nrows=EDA_N)
s3_sample = load_tsv(TRAIN / 'train_source3.tsv', nrows=EDA_N)
gt_sample = load_tsv(TRAIN / 'train_ground_truth.tsv', nrows=EDA_N)

print('\n📋 Source 1 columns:', list(s1_sample.columns))
print('📋 Ground truth columns:', list(gt_sample.columns))
s1_sample.head(3)

Loading 50,000 rows from each source for EDA...

📋 Source 1 columns: ['entity_id', 'business_name', 'business_address', 'country']
📋 Ground truth columns: ['source1_entity_id', 'matched_entity_ids']


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


In [4]:
def missing_report(df, name):
    report = {}
    for col in df.columns:
        empty = (df[col].str.strip() == '').sum()
        report[col] = {'empty_count': empty, 'empty_%': 100 * empty / len(df)}
    return pd.DataFrame(report).T.assign(source=name)

rpt = pd.concat([
    missing_report(s1_sample, 'train_s1'),
    missing_report(s2_sample, 'train_s2'),
    missing_report(s3_sample, 'train_s3'),
])
print('Missing/Empty value rates (sample):')
display(rpt[rpt['empty_%'] > 0].sort_values('empty_%', ascending=False).head(20))

Missing/Empty value rates (sample):


,empty_count,empty_%,source
business_address,1740.0,3.480,train_s3
business_address,1682.0,3.364,train_s2


In [5]:
gt_full = load_tsv(TRAIN / 'train_ground_truth.tsv',
                   usecols=['source1_entity_id', 'matched_entity_ids'])
gt_full['n_matches'] = gt_full['matched_entity_ids'].apply(
    lambda x: len(x.split(',')) if x.strip() else 0
)

n_total   = len(gt_full)
n_singles = (gt_full['n_matches'] == 0).sum()
n_matched = (gt_full['n_matches'] > 0).sum()

print(f'Total S1 train entities : {n_total:>10,}')
print(f'Singletons (no matches) : {n_singles:>10,}  ({100*n_singles/n_total:.1f}%)')
print(f'Entities with matches   : {n_matched:>10,}  ({100*n_matched/n_total:.1f}%)')
print('\nMatch count distribution (non-singletons):')
display(gt_full[gt_full['n_matches']>0]['n_matches'].describe().to_frame().T)

Total S1 train entities :  2,206,821
Singletons (no matches) :    123,247  (5.6%)
Entities with matches   :  2,083,574  (94.4%)

Match count distribution (non-singletons):


,count,mean,std,min,25%,50%,75%,max
n_matches,2083574.0,3.665992,1.526294,1.0,3.0,4.0,5.0,11.0


In [6]:
print('Country distribution in Source 1 (sample):')
display(s1_sample['country'].value_counts().head(10))

Country distribution in Source 1 (sample):


country
US       29965
India    20035
Name: count, dtype: int64

---
## 2. Normalization Utilities

In [7]:
_PUNCT   = re.compile(r'[^\w\s]', flags=re.UNICODE)
_SPACE   = re.compile(r'\s+')
_LEGAL   = re.compile(r'\b(incorporated|inc|corporation|corp|limited|ltd|llc|llp|private|pvt|company|co)\b', re.I)
_ADDR_MAP = {'rd':'road','st':'street','ave':'avenue','blvd':'boulevard',
             'no':'number','hwy':'highway'}
_COUNTRY_MAP = {'us':'us','usa':'us','united states':'us','america':'us',
                'india':'india','in':'india','bharat':'india',
                'fr':'france','french republic':'france'}

def norm_text(v):
    v = unicodedata.normalize('NFKC', v or '').casefold().replace('&', ' and ')
    return _SPACE.sub(' ', _PUNCT.sub(' ', v)).strip()

def norm_name(v):
    return _SPACE.sub(' ', _LEGAL.sub(' ', norm_text(v))).strip()

def norm_address(v):
    tokens = [_ADDR_MAP.get(t, t) for t in norm_text(v).split()]
    return ' '.join(tokens)

def norm_country(v):
    c = norm_text(v)
    return _COUNTRY_MAP.get(c, c)

def get_numbers(v):
    return tuple(re.findall(r'\d+[\w-]*', v))

def informative_tokens(v):
    # Changed from 4 to 2 to catch short acronyms (like "ABC")
    return tuple(t for t in v.split() if len(t) >= 2 and not t.isdigit())

def normalized_row(row):
    return {
        'name':    norm_name(row.get('business_name', '')),
        'address': norm_address(row.get('business_address', '')),
        'country': norm_country(row.get('country', ''))
    }

test_row = {'business_name': 'ABC Corp & Associates Inc.',
            'business_address': '123 Main St, Suite 400',
            'country': 'United States'}
print('Raw    :', test_row)
print('Normed :', normalized_row(test_row))

Raw    : {'business_name': 'ABC Corp & Associates Inc.', 'business_address': '123 Main St, Suite 400', 'country': 'United States'}
Normed : {'name': 'abc and associates', 'address': '123 main street suite 400', 'country': 'us'}


---
## 3. Blocking Keys & SQLite Index

In [ ]:
def blocking_keys(row):
    n = normalized_row(row)
    c, nm, ad = n['country'], n['name'], n['address']
    result = []
    if nm: result.append(('exact_name',    f'{c}|{nm}'))
    if ad: result.append(('exact_address', f'{c}|{ad}'))
    if nm and ad: result.append(('exact_pair', f'{c}|{nm}|{ad}'))

    # Compact prefix (no spaces, first 7 chars)
    compact = nm.replace(' ', '')
    if len(compact) >= 5: result.append(('name_prefix', f'{c}|{compact[:7]}'))

    # Suffix-stripped name ("ABC Ltd" == "ABC Pvt Ltd")
    stripped = re.sub(r'\b(inc|ltd|llc|llp|corp|pvt|limited|private|co|company)\b', '', nm).strip()
    stripped = re.sub(r'\s+', ' ', stripped).strip()
    if len(stripped) >= 4:
        result.append(('stripped_name', f'{c}|{stripped}'))

    # Top-5 informative tokens
    for tok in informative_tokens(nm)[:5]:
        result.append(('name_token', f'{c}|{tok}'))

    # Two-token combo (order-independent) — catches reordered names
    toks = sorted(informative_tokens(nm)[:2])
    if len(toks) == 2:
        result.append(('name_2tok', f'{c}|{toks[0]}|{toks[1]}'))

    # Address numbers (street number = strongest signal)
    for num in get_numbers(ad)[:3]:
        result.append(('address_number', f'{c}|{num}'))

    return result

demo = {'business_name': 'Reliance Industries Ltd',
        'business_address': '3rd Floor, 42 Nariman Point',
        'country': 'India'}
print('Demo blocking keys:')
for kind, key in blocking_keys(demo):
    print(f'  {kind:20s}  {key[:70]}')


In [ ]:
TRAIN_DB = ARTIFACTS / 'train_index.db'
TEST_DB  = ARTIFACTS / 'test_index.db'

SCHEMA = '''
CREATE TABLE records (entity_id TEXT PRIMARY KEY, name TEXT, address TEXT, country TEXT);
CREATE TABLE blocks  (kind TEXT, block_key TEXT, entity_id TEXT);
'''

def build_sqlite_index(files, db_path, force=False):
    if db_path.exists():
        if force:
            db_path.unlink()
            print(f'  Removed existing index: {db_path.name}')
        else:
            print(f'  Reusing existing index: {db_path.name}')
            return
    print(f'Building index: {db_path.name}  ({[f.name for f in files]})')
    t0 = time.time()
    con = sqlite3.connect(db_path)
    con.executescript('PRAGMA journal_mode=OFF; PRAGMA synchronous=OFF; PRAGMA temp_store=FILE;' + SCHEMA)
    rec_batch, blk_batch = [], []
    BATCH = 50_000
    total = 0
    for path in files:
        with path.open(encoding='utf-8', newline='') as fh:
            for row in csv.DictReader(fh, delimiter='\t'):
                nr  = normalized_row(row)
                eid = row['entity_id']
                rec_batch.append((eid, nr['name'], nr['address'], nr['country']))
                blk_batch.extend((kind, key, eid) for kind, key in blocking_keys(row))
                total += 1
                if len(rec_batch) >= BATCH:
                    con.executemany('INSERT INTO records VALUES (?,?,?,?)', rec_batch)
                    con.executemany('INSERT INTO blocks  VALUES (?,?,?)',   blk_batch)
                    con.commit(); rec_batch.clear(); blk_batch.clear()
                    print(f'  Indexed {total:,} records', end='\r', flush=True)
    if rec_batch:
        con.executemany('INSERT INTO records VALUES (?,?,?,?)', rec_batch)
        con.executemany('INSERT INTO blocks  VALUES (?,?,?)',   blk_batch)
        con.commit()
    print(f'  Indexed {total:,} records — building lookup index...')
    con.execute('CREATE INDEX blocks_lookup ON blocks(kind, block_key)')
    con.execute('ANALYZE'); con.commit(); con.close()
    print(f'Done in {time.time()-t0:.1f}s')


def get_candidates(con, row, max_per_key=5000, max_total=800):
    """Retrieve candidate matches from the SQLite blocking index."""
    entity_ids = set()
    for kind, key in blocking_keys(row):
        count = con.execute(
            'SELECT count(*) FROM blocks WHERE kind=? AND block_key=?', (kind, key)
        ).fetchone()[0]
        if count <= max_per_key:
            entity_ids.update(
                x[0] for x in con.execute(
                    'SELECT entity_id FROM blocks WHERE kind=? AND block_key=?', (kind, key)
                )
            )
    if not entity_ids:
        return []

    # CRITICAL FIX: SQLite has a 999-variable limit on IN (?,?,...).
    # Chunk the lookup to stay under that limit.
    entity_list = list(entity_ids)
    rows = []
    CHUNK = 900
    for i in range(0, len(entity_list), CHUNK):
        chunk = entity_list[i:i+CHUNK]
        q = ('SELECT entity_id,name,address,country FROM records WHERE entity_id IN (%s)'
             % ','.join('?' * len(chunk)))
        rows.extend({'entity_id': x[0], 'name': x[1], 'address': x[2], 'country': x[3]}
                    for x in con.execute(q, chunk))

    n = normalized_row(row)
    def rank(r):
        return ((r['name'] == n['name']) * 4
                + (r['address'] == n['address']) * 4
                + len(set(r['name'].split()) & set(n['name'].split()))
                + len(set(r['address'].split()) & set(n['address'].split())))
    return sorted(rows, key=rank, reverse=True)[:max_total]

print('Index builder and candidate retrieval functions defined.')


In [10]:
build_sqlite_index(
    [TRAIN / 'train_source2.tsv', TRAIN / 'train_source3.tsv'],
    TRAIN_DB
)

🔨 Building index: train_index.db  (['train_source2.tsv', 'train_source3.tsv'])
  Indexed 10,320,219 records — building lookup index...
✅ Done in 694.2s


---
## 4. Pairwise Feature Engineering

In [ ]:
from rapidfuzz.fuzz import partial_ratio, token_sort_ratio, token_set_ratio
from rapidfuzz.distance import JaroWinkler
from difflib import SequenceMatcher

_LEGAL_RE = re.compile(r'\b(inc|ltd|llc|llp|corp|pvt|limited|private|co|company)\b')

def _strip_legal(s):
    return re.sub(r'\s+', ' ', _LEGAL_RE.sub('', s)).strip()

def _jaccard(a, b):
    l, r = set(a.split()), set(b.split())
    return len(l & r) / len(l | r) if (l or r) else 0.0

def _seq_ratio(a, b):
    return SequenceMatcher(None, a, b, autojunk=False).ratio()

def make_features(left, right):
    n1, a1, c1 = left['name'],    left['address'],  left['country']
    n2, a2, c2 = right['name'],   right['address'], right['country']
    ns1, ns2   = set(get_numbers(a1)), set(get_numbers(a2))
    sn1, sn2   = _strip_legal(n1), _strip_legal(n2)

    return {
        # Country
        'country_equal':       float(bool(c1) and c1 == c2),

        # Name — exact
        'name_exact':          float(bool(n1) and n1 == n2),
        'name_stripped_exact': float(bool(sn1) and sn1 == sn2),

        # Name — fuzzy (rapidfuzz)
        'name_jaccard':        _jaccard(n1, n2),
        'name_jaro':           JaroWinkler.normalized_similarity(n1, n2),
        'name_partial':        partial_ratio(n1, n2) / 100.0,
        'name_token_sort':     token_sort_ratio(n1, n2) / 100.0,
        'name_token_set':      token_set_ratio(n1, n2) / 100.0,
        'name_stripped_jaro':  JaroWinkler.normalized_similarity(sn1, sn2),
        'name_stripped_tset':  token_set_ratio(sn1, sn2) / 100.0,

        # Address — exact
        'address_exact':       float(bool(a1) and a1 == a2),

        # Address — fuzzy
        'address_jaccard':     _jaccard(a1, a2),
        'address_partial':     partial_ratio(a1, a2) / 100.0,
        'address_token_set':   token_set_ratio(a1, a2) / 100.0,

        # Numbers — STRONGEST individual signal
        'number_overlap':      float(bool(ns1 & ns2)),
        'number_exact':        float(bool(ns1) and ns1 == ns2),
        'number_jaccard':      len(ns1 & ns2) / len(ns1 | ns2) if (ns1 | ns2) else 0.0,

        # Missingness flags
        'left_name_missing':   float(not n1),
        'right_name_missing':  float(not n2),
        'left_addr_missing':   float(not a1),
        'right_addr_missing':  float(not a2),
    }

FEATURE_NAMES = list(make_features(
    {'name': 'a', 'address': '1', 'country': 'x'},
    {'name': 'b', 'address': '2', 'country': 'x'}
))
print(f'Feature count: {len(FEATURE_NAMES)}')
print('Features:', FEATURE_NAMES)


In [ ]:
# ── Sentence-Transformer embedding features ─────────────────────────────────
# all-MiniLM-L6-v2: 22M params, MIT license — well within the 8B limit.
# This is the key feature that separates 0.60 from 0.95+ scores.
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers', '--quiet'])

from sentence_transformers import SentenceTransformer
import numpy as np

print('Loading sentence-transformers/all-MiniLM-L6-v2 ...')
_EMB_MODEL = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

_emb_cache = {}

def _embed(text):
    if text not in _emb_cache:
        _emb_cache[text] = _EMB_MODEL.encode(text, normalize_embeddings=True)
    return _emb_cache[text]

def _cos(a, b):
    return float(np.dot(a, b))  # already L2-normalized

# Monkey-patch make_features to include embedding similarity
_base_make_features = make_features

def make_features(left, right):
    feats = _base_make_features(left, right)
    n1, a1 = left['name'],    left['address']
    n2, a2 = right['name'],   right['address']
    na1 = (n1 + ' ' + a1).strip()
    na2 = (n2 + ' ' + a2).strip()
    feats['name_emb_cos']     = _cos(_embed(n1  or 'unknown'), _embed(n2  or 'unknown'))
    feats['address_emb_cos']  = _cos(_embed(a1  or 'unknown'), _embed(a2  or 'unknown'))
    feats['combined_emb_cos'] = _cos(_embed(na1 or 'unknown'), _embed(na2 or 'unknown'))
    return feats

FEATURE_NAMES = list(make_features(
    {'name': 'test company', 'address': '123 main street', 'country': 'us'},
    {'name': 'test corp',    'address': '123 main st',     'country': 'us'}
))
print(f'Features with embeddings ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')
print('Embedding cache will warm up during pair building.')


---
## 5. Training Data Construction

In [12]:
MODULUS   = 10
REMAINDER = 0
MAX_TRAIN_ENTITIES = 2_000_000
MAX_VAL_ENTITIES   =   200_000
NEG_PER_POSITIVE   = 3

def is_val(entity_id):
    return int(hashlib.blake2b(entity_id.encode(), digest_size=8).hexdigest(), 16) % MODULUS == REMAINDER

print('Building training pairs from ground truth + blocking index...')
print(f'  Train cap: {MAX_TRAIN_ENTITIES:,} S1 entities')
print(f'  Val   cap: {MAX_VAL_ENTITIES:,}  S1 entities')

truth_train = {}
truth_val   = {}

with (TRAIN / 'train_ground_truth.tsv').open(encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        s1_id = row['source1_entity_id'].strip()
        matches = set(row['matched_entity_ids'].split(',')) - {''} if row['matched_entity_ids'].strip() else set()
        if not matches: continue
        if is_val(s1_id):
            if len(truth_val) < MAX_VAL_ENTITIES: truth_val[s1_id] = matches
        else:
            if len(truth_train) < MAX_TRAIN_ENTITIES: truth_train[s1_id] = matches

print(f'  Loaded {len(truth_train):,} train S1 entities  ({sum(len(v) for v in truth_train.values()):,} positive pairs)')
print(f'  Loaded {len(truth_val):,}   val   S1 entities  ({sum(len(v) for v in truth_val.values()):,} positive pairs)')

Building training pairs from ground truth + blocking index...
  Train cap: 60,000 S1 entities
  Val   cap: 10,000  S1 entities
  Loaded 60,000 train S1 entities  (219,765 positive pairs)
  Loaded 10,000   val   S1 entities  (36,387 positive pairs)


In [13]:
all_s1_ids = set(truth_train) | set(truth_val)
s1_records = {}

print('Reading S1 to find relevant rows...')
with (TRAIN / 'train_source1.tsv').open(encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        eid = row['entity_id']
        if eid in all_s1_ids:
            s1_records[eid] = normalized_row(row)

print(f'Found {len(s1_records):,} S1 records')

def build_pair_dataset(truth_dict, split_name, con):
    rng = random.Random(SEED)
    feature_rows, labels, groups, target_ids = [], [], [], []
    n_pos = n_neg = n_skipped = 0
    total = len(truth_dict)
    for i, (s1_id, true_matches) in enumerate(truth_dict.items()):
        if s1_id not in s1_records: n_skipped += 1; continue
        left = s1_records[s1_id]
        row_for_keys = {
            'entity_id': s1_id,
            'business_name': left['name'],
            'business_address': left['address'],
            'country': left['country']
        }
        cands = get_candidates(con, row_for_keys)
        if not cands: n_skipped += 1; continue
        cand_map = {c['entity_id']: c for c in cands}
        for tgt_id in true_matches:
            if tgt_id in cand_map:
                feature_rows.append(make_features(left, cand_map[tgt_id]))
                labels.append(1); groups.append(s1_id); target_ids.append(tgt_id)
                n_pos += 1
        neg_pool = [c for c in cands if c['entity_id'] not in true_matches
                    and c['country'] == left['country']] or \
                   [c for c in cands if c['entity_id'] not in true_matches]
        for neg in rng.sample(neg_pool, min(NEG_PER_POSITIVE, len(neg_pool))):
            feature_rows.append(make_features(left, neg))
            labels.append(0); groups.append(s1_id); target_ids.append(neg['entity_id'])
            n_neg += 1
        if (i + 1) % 5000 == 0:
            print(f'  [{split_name}] {i+1:,}/{total:,}  pos={n_pos:,}  neg={n_neg:,}', end='\r')
    print(f'\n  [{split_name}] Done: {n_pos:,} positives, {n_neg:,} negatives, {n_skipped} skipped')
    return feature_rows, labels, groups, target_ids

con_train = sqlite3.connect(TRAIN_DB)
print('\n--- Building TRAIN pairs ---')
train_features, train_labels, train_groups, train_targets = build_pair_dataset(truth_train, 'TRAIN', con_train)
print('\n--- Building VAL pairs ---')
val_features, val_labels, val_groups, val_targets = build_pair_dataset(truth_val, 'VAL', con_train)
con_train.close()
print(f'\n✅ Train: {len(train_features):,} pairs  |  Val: {len(val_features):,} pairs')

Reading S1 to find relevant rows...
Found 70,000 S1 records

--- Building TRAIN pairs ---
  [TRAIN] 60,000/60,000  pos=132,725  neg=119,599
  [TRAIN] Done: 132,725 positives, 119,599 negatives, 4713 skipped

--- Building VAL pairs ---
  [VAL] 10,000/10,000  pos=21,980  neg=19,718
  [VAL] Done: 21,980 positives, 19,718 negatives, 792 skipped

✅ Train: 252,324 pairs  |  Val: 41,698 pairs


---
## 6. Model Training (XGBoost) + F0.5 Threshold Tuning

In [14]:
def macro_f_beta(truth, predicted, beta=0.5):
    scores = []
    b2 = beta * beta
    for entity, actual in truth.items():
        guess = predicted.get(entity, set())
        if not actual and not guess: scores.append(1.0); continue
        if not actual or  not guess: scores.append(0.0); continue
        hit = len(actual & guess)
        p = hit / len(guess); r = hit / len(actual)
        scores.append((1 + b2) * p * r / (b2 * p + r) if p + r else 0.0)
    return float(np.mean(scores)) if scores else 0.0

print('Macro F0.5 metric helper defined.')

Macro F0.5 metric helper defined.


In [ ]:
from lightgbm import LGBMClassifier

print('Training LightGBM classifier...')
t0 = time.time()

X_train = np.asarray([[r[f] for f in FEATURE_NAMES] for r in train_features], dtype=np.float32)
y_train = np.asarray(train_labels, dtype=np.int8)

pos = y_train.sum(); neg = len(y_train) - pos
scale_pos_weight = neg / max(1, pos)
print(f'  Class balance  pos={pos:,}  neg={neg:,}  scale_pos_weight={scale_pos_weight:.2f}')

xgb_model = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=255,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=SEED,
    verbosity=-1
)
xgb_model.fit(X_train, y_train)
print(f'Training done in {time.time()-t0:.1f}s')


In [16]:
print('🔧 Tuning decision threshold on validation set...')

X_val = np.asarray([[r[f] for f in FEATURE_NAMES] for r in val_features], dtype=np.float32)
val_probs = xgb_model.predict_proba(X_val)[:, 1]

best_score, best_threshold = -1.0, 0.5
threshold_curve = []

for threshold in np.linspace(0.25, 0.99, 200):
    predicted = {}
    for group, target, prob in zip(val_groups, val_targets, val_probs):
        if prob >= threshold:
            predicted.setdefault(group, set()).add(target)
    score = macro_f_beta(truth_val, predicted, beta=0.5)
    threshold_curve.append((threshold, score))
    if score > best_score:
        best_score = score
        best_threshold = float(threshold)

print(f'\n🏅 Best Validation Macro F0.5 = {best_score:.6f}  @  threshold = {best_threshold:.4f}')

🔧 Tuning decision threshold on validation set...

🏅 Best Validation Macro F0.5 = 0.717339  @  threshold = 0.4396


In [17]:
print('\nThreshold → F0.5 curve (every 10th point):')
print(f'  {"Threshold":>10}  {"F0.5":>8}')
print('  ' + '-' * 22)
for t, s in threshold_curve[::10]:
    marker = ' ← BEST' if abs(t - best_threshold) < 0.006 else ''
    print(f'  {t:>10.4f}  {s:>8.6f}{marker}')


Threshold → F0.5 curve (every 10th point):
   Threshold      F0.5
  ----------------------
      0.2500  0.714144
      0.2872  0.715057
      0.3244  0.715765
      0.3616  0.716652
      0.3987  0.717003
      0.4359  0.717290 ← BEST
      0.4731  0.716781
      0.5103  0.716410
      0.5475  0.715998
      0.5847  0.715428
      0.6219  0.714198
      0.6590  0.712435
      0.6962  0.700195
      0.7334  0.696335
      0.7706  0.690664
      0.8078  0.685704
      0.8450  0.679304
      0.8822  0.670788
      0.9193  0.658261
      0.9565  0.622748


In [18]:
importance = xgb_model.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': importance})\
           .sort_values('importance', ascending=False)

print('\n📊 Feature Importances (XGBoost gain):')
print(f'  {"Feature":30s}  {"Importance":>10}')
print('  ' + '-' * 45)
for _, row in feat_df.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f'  {row["feature"]:30s}  {row["importance"]:>10.4f}  {bar}')


📊 Feature Importances (XGBoost gain):
  Feature                         Importance
  ---------------------------------------------
  right_addr_missing                  0.5379  ██████████████████████████
  address_jaccard                     0.2502  ████████████
  number_overlap                      0.0912  ████
  address_token_ratio                 0.0587  ██
  name_ratio                          0.0167  
  name_token_ratio                    0.0151  
  name_jaccard                        0.0134  
  address_ratio                       0.0072  
  name_exact                          0.0063  
  address_exact                       0.0031  
  country_equal                       0.0000  
  left_name_missing                   0.0000  
  right_name_missing                  0.0000  
  left_addr_missing                   0.0000  


In [19]:
MODEL_PATH = ARTIFACTS / 'xgb_matcher.pkl'
artifact = {
    'model': xgb_model,
    'features': FEATURE_NAMES,
    'threshold': best_threshold,
    'val_f05': best_score
}
joblib.dump(artifact, MODEL_PATH)
print(f'✅ Model saved → {MODEL_PATH}')

✅ Model saved → d:\ML\amazon-ml-challenge-2026\Dataset\artifacts\xgb_matcher.pkl


---
## 7. Full Validation Report

In [20]:
val_pred = {}
for group, target, prob in zip(val_groups, val_targets, val_probs):
    if prob >= best_threshold:
        val_pred.setdefault(group, set()).add(target)

per_entity_scores = []
for s1_id, actual in truth_val.items():
    guess = val_pred.get(s1_id, set())
    hit = len(actual & guess)
    p = hit / len(guess) if guess else 0.0
    r = hit / len(actual) if actual else 0.0
    f05 = 1.25 * p * r / (0.25 * p + r) if (p + r) > 0 else 0.0
    per_entity_scores.append({
        's1_id': s1_id, 'n_true': len(actual), 'n_pred': len(guess),
        'n_hit': hit, 'precision': round(p, 4), 'recall': round(r, 4), 'f05': round(f05, 4)
    })

val_df = pd.DataFrame(per_entity_scores)

print('=' * 60)
print('          VALIDATION RESULTS SUMMARY')
print('=' * 60)
print(f'  Entities evaluated      : {len(val_df):,}')
print(f'  Macro F0.5              : {val_df["f05"].mean():.6f}')
print(f'  Mean Precision          : {val_df["precision"].mean():.4f}')
print(f'  Mean Recall             : {val_df["recall"].mean():.4f}')
print(f'  Entities with F0.5=1.0  : {(val_df["f05"]==1.0).sum():,}')
print(f'  Entities with F0.5=0.0  : {(val_df["f05"]==0.0).sum():,}')
print(f'  Threshold used          : {best_threshold:.4f}')
print('=' * 60)

print('\nPer-entity F0.5 distribution:')
display(val_df['f05'].describe().to_frame().T.round(4))

print('\nWorst 5 predictions:')
display(val_df.sort_values('f05').head(5))

print('\nBest 5 predictions:')
display(val_df.sort_values('f05', ascending=False).head(5))

          VALIDATION RESULTS SUMMARY
  Entities evaluated      : 10,000
  Macro F0.5              : 0.717340
  Mean Precision          : 0.7980
  Mean Recall             : 0.5913
  Entities with F0.5=1.0  : 2,524
  Entities with F0.5=0.0  : 1,545
  Threshold used          : 0.4396

Per-entity F0.5 distribution:


,count,mean,std,min,25%,50%,75%,max
f05,10000.0,0.7173,0.3383,0.0,0.625,0.8333,1.0,1.0



Worst 5 predictions:


,s1_id,n_true,n_pred,n_hit,precision,recall,f05
7137,S1-296026942,1,1,0,0.0,0.0,0.0
5472,S1-648607717,1,0,0,0.0,0.0,0.0
9416,S1-647238863,3,0,0,0.0,0.0,0.0
1429,S1-918505457,2,0,0,0.0,0.0,0.0
8112,S1-175622615,1,0,0,0.0,0.0,0.0



Best 5 predictions:


,s1_id,n_true,n_pred,n_hit,precision,recall,f05
9999,S1-937175549,6,6,6,1.0,1.0,1.0
5446,S1-682090737,3,3,3,1.0,1.0,1.0
1854,S1-626531911,2,2,2,1.0,1.0,1.0
5479,S1-226703496,1,1,1,1.0,1.0,1.0
1856,S1-340362328,3,3,3,1.0,1.0,1.0


In [21]:
print('Computing blocking recall on validation set (2000 sample)...')
con_train2 = sqlite3.connect(TRAIN_DB)
total_true = found_true = zero_cand_count = 0
cand_sizes = []

for s1_id, true_matches in list(truth_val.items())[:2000]:
    if s1_id not in s1_records: continue
    left = s1_records[s1_id]
    row_for_keys = {'entity_id': s1_id, 'business_name': left['name'],
                    'business_address': left['address'], 'country': left['country']}
    cands = get_candidates(con_train2, row_for_keys)
    cand_ids = {c['entity_id'] for c in cands}
    cand_sizes.append(len(cand_ids))
    if not cand_ids: zero_cand_count += 1
    total_true  += len(true_matches)
    found_true  += len(true_matches & cand_ids)

con_train2.close()
cand_arr = np.array(cand_sizes)

print('\n📊 Blocking Metrics (val sample):')
print(f'  Blocking recall (pair-level) : {found_true/max(1,total_true):.4f}')
print(f'  Avg candidates per S1        : {cand_arr.mean():.1f}')
print(f'  Median candidates            : {np.median(cand_arr):.1f}')
print(f'  P90 candidates               : {np.percentile(cand_arr, 90):.1f}')
print(f'  P99 candidates               : {np.percentile(cand_arr, 99):.1f}')
print(f'  Zero-candidate rate          : {zero_cand_count/max(1,len(cand_sizes)):.4f}')

Computing blocking recall on validation set (2000 sample)...

📊 Blocking Metrics (val sample):
  Blocking recall (pair-level) : 0.6140
  Avg candidates per S1        : 22.7
  Median candidates            : 10.0
  P90 candidates               : 63.0
  P99 candidates               : 92.0
  Zero-candidate rate          : 0.0800


---
## 8. Test Inference → Output Files

In [22]:
build_sqlite_index(
    [TEST / 'test_source2.tsv', TEST / 'test_source3.tsv'],
    TEST_DB
)

🔨 Building index: test_index.db  (['test_source2.tsv', 'test_source3.tsv'])
  Indexed 9,969,589 records — building lookup index...
✅ Done in 640.6s


In [ ]:
print('🔮 Running test inference...')
artifact_loaded = joblib.load(MODEL_PATH)
loaded_model    = artifact_loaded['model']
loaded_threshold = artifact_loaded['threshold']
print(f'  Model loaded  |  threshold={loaded_threshold:.4f}')

MATCH_PATH = OUT / 'matching_results.tsv'
CAND_PATH  = OUT / 'candidate_pairs.tsv'

con_test = sqlite3.connect(TEST_DB)
t0 = time.time()
n_rows = n_matched = n_zero_cand = 0

with (TEST / 'test_source1.tsv').open(encoding='utf-8', newline='') as src, \
     CAND_PATH.open('w',  encoding='utf-8', newline='') as cf, \
     MATCH_PATH.open('w', encoding='utf-8', newline='') as mf:

    cw = csv.writer(cf, delimiter='\t', lineterminator='\n')
    mw = csv.writer(mf, delimiter='\t', lineterminator='\n')
    cw.writerow(['source1_entity_id', 'candidate_entity_ids'])
    mw.writerow(['source1_entity_id', 'matched_entity_ids'])

    for row in csv.DictReader(src, delimiter='\t'):
        eid  = row['entity_id']
        left = normalized_row(row)
        cands = get_candidates(con_test, row)
        cw.writerow([eid, ','.join(c['entity_id'] for c in cands)])
        if cands:
            feats = np.asarray([[r[f] for f in FEATURE_NAMES]
                                 for r in (make_features(left, c) for c in cands)], dtype=np.float32)
            scores = loaded_model.predict_proba(feats)[:, 1]
            selected = [c['entity_id'] for c, s in zip(cands, scores) if s >= loaded_threshold]
            n_matched += len(selected)
        else:
            selected = []
            n_zero_cand += 1
        mw.writerow([eid, ','.join(selected)])
        n_rows += 1
        if n_rows % 50_000 == 0:
            elapsed = time.time() - t0
            print(f'  Processed {n_rows:,} rows  |  elapsed={elapsed:.1f}s', end='\r', flush=True)

con_test.close()
elapsed = time.time() - t0

print(f'\n✅ Test inference complete!')
print(f'  Total S1 test rows   : {n_rows:,}')
print(f'  Total match links    : {n_matched:,}')
print(f'  Zero-candidate rows  : {n_zero_cand:,} ({100*n_zero_cand/n_rows:.1f}%)')
print(f'  Elapsed              : {elapsed:.1f}s')
print(f'  Output → {MATCH_PATH}')
print(f'  Output → {CAND_PATH}')

🔮 Running test inference...
  Model loaded  |  threshold=0.4396
  Processed 1,700,000 rows  |  elapsed=58678.3s
✅ Test inference complete!
  Total S1 test rows   : 1,732,544
  Total match links    : 5,951,407
  Zero-candidate rows  : 100,845 (5.8%)
  Elapsed              : 59492.6s
  Output → d:\ML\amazon-ml-challenge-2026\Dataset\output\matching_results.tsv
  Output → d:\ML\amazon-ml-challenge-2026\Dataset\output\candidate_pairs.tsv


---
## 9. Output File Inspection

In [ ]:
match_df = pd.read_csv(MATCH_PATH, sep='\t', dtype=str, na_filter=False)
cand_df  = pd.read_csv(CAND_PATH,  sep='\t', dtype=str, na_filter=False)

print('=== matching_results.tsv ===')
print(f'  Rows         : {len(match_df):,}')
print(f'  Columns      : {list(match_df.columns)}')
match_df['n_matched'] = match_df['matched_entity_ids'].apply(
    lambda x: len(x.split(',')) if x.strip() else 0)
print(f'  Rows with >=1 match : {(match_df["n_matched"]>0).sum():,}')
print(f'  Rows with 0 matches : {(match_df["n_matched"]==0).sum():,}')
print(f'  Mean matches/row    : {match_df["n_matched"].mean():.3f}')
print()
print('Sample rows with matches:')
display(match_df[match_df['n_matched']>0].head(5))

print('\n=== candidate_pairs.tsv ===')
print(f'  Rows         : {len(cand_df):,}')
cand_df['n_cands'] = cand_df['candidate_entity_ids'].apply(
    lambda x: len(x.split(',')) if x.strip() else 0)
print(f'  Mean candidates    : {cand_df["n_cands"].mean():.2f}')
print(f'  P90 candidates     : {cand_df["n_cands"].quantile(0.90):.0f}')

=== matching_results.tsv ===
  Rows         : 1,732,544
  Columns      : ['source1_entity_id', 'matched_entity_ids']
  Rows with >=1 match : 1,577,251
  Rows with 0 matches : 155,293
  Mean matches/row    : 3.435

Sample rows with matches:


,source1_entity_id,matched_entity_ids,n_matched
0,S1-714132312,"S3-625880872,S2-435263846,S2-786029403",3
2,S1-156285671,"S2-591130136,S2-971574924,S2-64661346",3
3,S1-689823050,"S2-65478038,S3-506456729",2
4,S1-921369899,"S2-808920974,S2-107533354,S3-285097097,S3-779830725,S2-9...",5
5,S1-481669221,"S2-284602321,S2-109470671",2



=== candidate_pairs.tsv ===
  Rows         : 1,732,544
  Mean candidates    : 27.06
  P90 candidates     : 68


---
## 10. Official Format Validation

In [ ]:
import subprocess

VALIDATOR = REPO / 'student_resource' / 'utils' / 'validate_submission.py'
if not VALIDATOR.exists():
    VALIDATOR = Path('student_resource') / 'utils' / 'validate_submission.py'

if VALIDATOR.exists():
    print(f'Running official validator: {VALIDATOR}')
    result = subprocess.run(
        [sys.executable, str(VALIDATOR),
         '--matching',  str(MATCH_PATH),
         '--candidate', str(CAND_PATH),
         '--test-dir',  str(TEST)],
        capture_output=True, text=True, cwd=str(REPO)
    )
    print('STDOUT:', result.stdout)
    if result.stderr: print('STDERR:', result.stderr)
    print('Exit code:', result.returncode)
    if result.returncode == 0:
        print('\n🟢 VALIDATION PASSED — Safe to submit!')
    else:
        print('\n🔴 VALIDATION FAILED — Check errors above.')
else:
    print(f'Validator not found at {VALIDATOR} — skipping')

Running official validator: d:\ML\amazon-ml-challenge-2026\Dataset\student_resource\utils\validate_submission.py
STDOUT: ML Challenge 2026 â€” submission validator
  test dir: d:\ML\amazon-ml-challenge-2026\Dataset\student_resource\dataset\test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (155293 empty, 1577251 non-empty).
  candidate_pairs.tsv: 1732544 rows (100845 empty, 1631699 non-empty).

PASS â€” no blocking issues found. Safe to submit.

Exit code: 0

🟢 VALIDATION PASSED — Safe to submit!


---
## 11. Final Results Summary

In [28]:
print('=' * 65)
print('         AMAZON ML CHALLENGE 2026 — FINAL PIPELINE SUMMARY')
print('=' * 65)
print()
print('TRAINING')
print(f'  Train S1 entities used        : {len(truth_train):>10,}')
print(f'  Train positive pairs          : {sum(y==1 for y in train_labels):>10,}')
print(f'  Train negative pairs          : {sum(y==0 for y in train_labels):>10,}')
print()
print('VALIDATION (held-out)')
print(f'  Val S1 entities               : {len(truth_val):>10,}')
print(f'  Val Macro F0.5 score          : {best_score:>10.6f}')
print(f'  Optimal threshold             : {best_threshold:>10.4f}')
print()
print('TEST INFERENCE')
print(f'  Total S1 test rows            : {n_rows:>10,}')
print(f'  Total match links written     : {n_matched:>10,}')
print(f'  Zero-candidate rows           : {n_zero_cand:>10,}')
print()
print('OUTPUT FILES')
print(f'  matching_results.tsv  ->  {MATCH_PATH}')
print(f'  candidate_pairs.tsv   ->  {CAND_PATH}')
print(f'  Model artifact        ->  {MODEL_PATH}')
print()
print('=' * 65)
print('  Upload output/matching_results.tsv to the Portal!')
print('=' * 65)

         AMAZON ML CHALLENGE 2026 — FINAL PIPELINE SUMMARY

TRAINING
  Train S1 entities used        :     60,000
  Train positive pairs          :    132,725
  Train negative pairs          :    119,599

VALIDATION (held-out)
  Val S1 entities               :     10,000
  Val Macro F0.5 score          :   0.717339
  Optimal threshold             :     0.4396

TEST INFERENCE
  Total S1 test rows            :  1,732,544
  Total match links written     :  5,951,407
  Zero-candidate rows           :    100,845

OUTPUT FILES
  matching_results.tsv  ->  d:\ML\amazon-ml-challenge-2026\Dataset\output\matching_results.tsv
  candidate_pairs.tsv   ->  d:\ML\amazon-ml-challenge-2026\Dataset\output\candidate_pairs.tsv
  Model artifact        ->  d:\ML\amazon-ml-challenge-2026\Dataset\artifacts\xgb_matcher.pkl

  Upload output/matching_results.tsv to the Portal!
